# Unidad 1: Arquitecturas Big Data y ETL batch distribuido

## Sesión 3: Procesamiento y calidad de datos
### Filtrado, duplicados, nulos y particionamiento analítico

### Objetivo
Validar la estructura y calidad de los datos, aplicar
tratamientos justificados y guardar una salida analítica
particionada en Parquet.

### Datos de la práctica guiada
Utilizaremos customers.csv y articles.csv del dataset H&M,
copiados desde la sesión 2 y verificados contra los originales.

### Actividades
- Validar el esquema y las columnas requeridas.
- Detectar y tratar valores nulos y duplicados.
- Filtrar y ordenar registros.
- Escribir y leer resultados particionados.
- Comprobar los conteos y el plan de ejecución.
- Identificar las capas Bronze, Silver y Gold.

### Aplicación al Proyecto Sello
Después aplicaremos los controles de calidad y la escritura
analítica a los datos de AgroSmart Puno.

### Alcance
Trabajaremos en el entorno local lambda26.
Esta sesión no instala un clúster HDFS.

## 3.2 Crear la SparkSession

Crearemos una sesión de Spark para trabajar la calidad
de los datos.

Configuración indicada por el docente:
- Aplicación: sesion3-calidad-datos.
- Ejecución: local[*].
- Puerto interno de Spark UI: 4040.
- Particiones de shuffle: 8.
- Memoria del driver: 4 GB.

Spark UI será accesible desde http://localhost:4042.
Todavía no cargaremos los archivos.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion3-calidad-datos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 22:11:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Definir las rutas de trabajo

Utilizaremos dos variables:
- ORIGEN_DATOS: carpeta con customers.csv y articles.csv.
- ARTIFACTS: carpeta destinada a los resultados de S03.

Las rutas corresponden al interior del contenedor.
Así mantenemos separados los datos y resultados
de cada sesión.

In [2]:
ORIGEN_DATOS = "/opt/s03-procesamiento-calidad-datos/data"
ARTIFACTS = "/opt/s03-procesamiento-calidad-datos/artifacts"

In [3]:
f"{ORIGEN_DATOS}/customers.csv"

'/opt/s03-procesamiento-calidad-datos/data/customers.csv'

## 3.3 Cargar customers.csv y validar el esquema

### Definir la estructura esperada

Utilizaremos el mismo esquema explícito de S02:
- customer_id, club_member_status, fashion_news_frequency
  y postal_code: texto.
- FN y Active: números decimales.
- age: número entero.

Definir los tipos permite controlar la lectura, pero no
demuestra por sí solo que el archivo tenga todos los campos
correctos ni que sus valores sean válidos.


In [5]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

schema_customers = StructType([
    StructField("customer_id", StringType(), nullable=True),
    StructField("FN", DoubleType(), nullable=True),
    StructField("Active", DoubleType(), nullable=True),
    StructField("club_member_status", StringType(), nullable=True),
    StructField("fashion_news_frequency", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
    StructField("postal_code", StringType(), nullable=True),
])

### Leer el archivo de clientes

Cargaremos customers.csv utilizando schema_customers.

header=True indica que la primera fila contiene encabezados.
schema establece los nombres y tipos esperados.

Después mostraremos la estructura que Spark asignó
al DataFrame.

In [6]:
df_customers = spark.read.csv(
    f"{ORIGEN_DATOS}/customers.csv",
    header=True,
    schema=schema_customers,
)

df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: double (nullable = true)
 |-- club_member_status: string (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)



### Comprobar las columnas requeridas

Compararemos los nombres de las columnas del DataFrame
con el conjunto de columnas esperado.

Si falta alguna, detendremos este paso con un mensaje de error.

Esta comprobación revisa el DataFrame construido. Como usamos
un esquema explícito, no sustituye la revisión del encabezado
original del CSV ni de los valores leídos.

In [7]:
columnas_requeridas = {
    "customer_id",
    "FN",
    "Active",
    "club_member_status",
    "fashion_news_frequency",
    "age",
    "postal_code",
}

faltantes = columnas_requeridas - set(df_customers.columns)

if faltantes:
    raise ValueError(
        f"Faltan columnas obligatorias: {sorted(faltantes)}"
    )

print("Esquema validado: las 7 columnas requeridas estan presentes.")

Esquema validado: las 7 columnas requeridas estan presentes.


### Confirmar las columnas y el conteo de filas

Mostraremos los nombres de las columnas en su orden
y contaremos todos los registros del DataFrame.

Compararemos el conteo con S02, donde obtuvimos
1 371 980 clientes.

In [8]:
print(df_customers.columns)

df_customers.count()

['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']


1371980

## 3.4 Explorar nulos por columna

Contaremos cuántos valores nulos contiene cada columna.

Un valor nulo representa información ausente.
No equivale automáticamente a cero ni a una cadena vacía.

Primero identificaremos los faltantes; todavía no
modificaremos los datos.

In [9]:
from pyspark.sql.functions import col, count, when

total_filas = df_customers.count()

df_customers.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_customers.columns
]).show(vertical=True, truncate=False)

[Stage 6:==>                                                      (1 + 19) / 20]

-RECORD 0------------------------
 customer_id            | 0      
 FN                     | 895050 
 Active                 | 907576 
 club_member_status     | 6062   
 fashion_news_frequency | 16009  
 age                    | 15861  
 postal_code            | 0      



### Calcular el porcentaje de valores nulos

Convertiremos los conteos de nulos a porcentajes
respecto del total de registros.

Esto permite comparar qué proporción de información
falta en cada columna, sin modificar los datos.

In [10]:
nulos = df_customers.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_customers.columns
]).collect()[0].asDict()

for columna, cantidad in nulos.items():
    porcentaje = cantidad / total_filas * 100
    print(f"{columna}: {cantidad} nulos ({porcentaje:.1f}%)")

customer_id: 0 nulos (0.0%)
FN: 895050 nulos (65.2%)
Active: 907576 nulos (66.2%)
club_member_status: 6062 nulos (0.4%)
fashion_news_frequency: 16009 nulos (1.2%)
age: 15861 nulos (1.2%)
postal_code: 0 nulos (0.0%)


### Detectar identificadores vacíos

customer_id es una columna crítica para identificar clientes.

Además de los nulos, comprobaremos si contiene textos vacíos
o compuestos únicamente por espacios.

Utilizaremos trim() para retirar los espacios de los extremos
antes de comparar con una cadena vacía.

In [11]:
from pyspark.sql.functions import trim

df_customers.select(
    count(
        when(col("customer_id").isNull(), 1)
    ).alias("identificadores_nulos"),

    count(
        when(trim(col("customer_id")) == "", 1)
    ).alias("identificadores_vacios")
).show()

+---------------------+----------------------+
|identificadores_nulos|identificadores_vacios|
+---------------------+----------------------+
|                    0|                     0|
+---------------------+----------------------+



## 3.5 Filtrado de datos

### Filtrar mediante una expresión SQL

Seleccionaremos los clientes cuya edad sea mayor que 30.

filter() conserva las filas que cumplen la condición.
show(3) solicita y muestra tres filas del resultado.

El DataFrame original no se modifica.

In [12]:
df_customers.filter("age > 30").show(3)

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


### Filtrar por estado de membresía

Seleccionaremos las filas donde club_member_status
sea exactamente ACTIVE.

Esta condición se refiere al estado de membresía,
no a la columna distinta llamada Active.

In [13]:
df_customers.filter("club_member_status = 'ACTIVE'").show()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|000058a12d5b43e67...|NULL|  NULL|            ACTIVE|                  NONE| 24|64f17e6a330a85798...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
|0000757967448a6cb...|NULL|  NULL|            ACTIVE|                  NONE| 20|fe7b8e2b3fafb89ca...|
|00007d2de826758b6...| 1.0|   1.0|            ACTIVE|             Regularly| 32|8d

### Filtrar un rango de edades con BETWEEN

Seleccionaremos clientes con edades entre 25 y 35 años.

BETWEEN incluye ambos límites: las edades 25 y 35
también cumplen la condición.

In [14]:
df_customers.filter("age BETWEEN 25 AND 35").show()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|00007d2de826758b6...| 1.0|   1.0|            ACTIVE|             Regularly| 32|8d6f45050876d059c...|
|0000945f66de1a11d...|NULL|  NULL|            ACTIVE|                  NONE| 29|d93e1aaecdebf9f71...|
|000097d91384a0c14...|NULL|  NULL|            ACTIVE|                  NONE| 31|2c29ae653a9282cce...|
|0000ae1bbb25e04bd...| 1.0|   1.0|            ACTIVE|             Regularly| 29|2c29ae653a9282cce...|
|0000e9a4db2da4e9c...| 1.0|   1.0|            ACTIVE|             Regularly| 27|de497104dda880ac0...|
|0000f1c71aafe5963...| 1.0|   1.0|            ACTIVE|             Regularly| 30|7c

### Filtrar varias categorías con IN

Seleccionaremos clientes cuyo valor de fashion_news_frequency
sea Regularly o Monthly.

IN permite comprobar si un valor pertenece a una lista
de opciones.

In [15]:
df_customers.filter(
    "fashion_news_frequency IN ('Regularly', 'Monthly')"
).show()

+--------------------+---+------+------------------+----------------------+---+--------------------+
|         customer_id| FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+---+------+------------------+----------------------+---+--------------------+
|00006413d8573cd20...|1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
|00007d2de826758b6...|1.0|   1.0|            ACTIVE|             Regularly| 32|8d6f45050876d059c...|
|00009d946eec3ea54...|1.0|   1.0|            ACTIVE|             Regularly| 56|b31984b20a8c478de...|
|0000ae1bbb25e04bd...|1.0|   1.0|            ACTIVE|             Regularly| 29|2c29ae653a9282cce...|
|0000b2f1829e23b24...|1.0|   1.0|            ACTIVE|             Regularly| 54|ca8ca81e8b5794992...|
|0000b95f630aaa931...|1.0|   1.0|            ACTIVE|             Regularly| 49|f63abf76506122d9f...|
|0000d6c053fc8f938...|1.0|   1.0|            ACTIVE|             Regularly| 41|5b5f53c673d0

### Filtrar utilizando col()

Seleccionaremos clientes mayores de 30 años mediante
una expresión de columna.

Esta condición es equivalente a la expresión SQL "age > 30".

In [16]:
from pyspark.sql.functions import col

df_customers.filter(col("age") > 30).show()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
|00007d2de826758b6...| 1.0|   1.0|            ACTIVE|             Regularly| 32|8d6f45050876d059c...|
|000097d91384a0c14...|NULL|  NULL|            ACTIVE|                  NONE| 31|2c29ae653a9282cce...|
|00009c2aeae8761f7...|NULL|  NULL|            ACTIVE|                  NONE| 49|7e2caa18837edc6a7...|
|00009d946eec3ea54...| 1.0|   1.0|            ACTIVE|             Regularly| 56|b3

### Comparar texto mediante col()

Seleccionaremos clientes cuyo club_member_status sea ACTIVE.

En una expresión de columna utilizamos == para comparar.
La condición es equivalente al filtro SQL anterior.

In [17]:
df_customers.filter(
    col("club_member_status") == "ACTIVE"
).show()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|000058a12d5b43e67...|NULL|  NULL|            ACTIVE|                  NONE| 24|64f17e6a330a85798...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
|0000757967448a6cb...|NULL|  NULL|            ACTIVE|                  NONE| 20|fe7b8e2b3fafb89ca...|
|00007d2de826758b6...| 1.0|   1.0|            ACTIVE|             Regularly| 32|8d

### Filtrar un rango mediante between()

Seleccionaremos clientes de 25 a 35 años utilizando
una expresión de columna.

between(25, 35) incluye ambos límites y equivale
a la expresión SQL age BETWEEN 25 AND 35.

In [18]:
df_customers.filter(
    col("age").between(25, 35)
).show()


+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|00007d2de826758b6...| 1.0|   1.0|            ACTIVE|             Regularly| 32|8d6f45050876d059c...|
|0000945f66de1a11d...|NULL|  NULL|            ACTIVE|                  NONE| 29|d93e1aaecdebf9f71...|
|000097d91384a0c14...|NULL|  NULL|            ACTIVE|                  NONE| 31|2c29ae653a9282cce...|
|0000ae1bbb25e04bd...| 1.0|   1.0|            ACTIVE|             Regularly| 29|2c29ae653a9282cce...|
|0000e9a4db2da4e9c...| 1.0|   1.0|            ACTIVE|             Regularly| 27|de497104dda880ac0...|
|0000f1c71aafe5963...| 1.0|   1.0|            ACTIVE|             Regularly| 30|7c

### Filtrar categorías mediante isin()

Seleccionaremos clientes cuyo fashion_news_frequency
sea Regularly o Monthly.

isin() comprueba si el valor pertenece a las opciones
indicadas. Equivale al filtro SQL con IN.

In [19]:
df_customers.filter(
    col("fashion_news_frequency").isin("Regularly", "Monthly")
).show()

+--------------------+---+------+------------------+----------------------+---+--------------------+
|         customer_id| FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+---+------+------------------+----------------------+---+--------------------+
|00006413d8573cd20...|1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
|00007d2de826758b6...|1.0|   1.0|            ACTIVE|             Regularly| 32|8d6f45050876d059c...|
|00009d946eec3ea54...|1.0|   1.0|            ACTIVE|             Regularly| 56|b31984b20a8c478de...|
|0000ae1bbb25e04bd...|1.0|   1.0|            ACTIVE|             Regularly| 29|2c29ae653a9282cce...|
|0000b2f1829e23b24...|1.0|   1.0|            ACTIVE|             Regularly| 54|ca8ca81e8b5794992...|
|0000b95f630aaa931...|1.0|   1.0|            ACTIVE|             Regularly| 49|f63abf76506122d9f...|
|0000d6c053fc8f938...|1.0|   1.0|            ACTIVE|             Regularly| 41|5b5f53c673d0

### Comparar rangos inclusivos y estrictos

between(25, 35) incluye las edades 25 y 35.

La condición (age > 25) y (age < 35) excluye ambas edades.
Compararemos los conteos para comprobar la diferencia.

In [20]:
print(
    "between (incluye 25 y 35):",
    df_customers.filter(col("age").between(25, 35)).count()
)

print(
    "estricto (excluye 25 y 35):",
    df_customers.filter(
        (col("age") > 25) & (col("age") < 35)
    ).count()
)

between (incluye 25 y 35): 413179
estricto (excluye 25 y 35): 338303


### Comparar valores con eqNullSafe()

Utilizaremos eqNullSafe() para buscar membresías ACTIVE
y para identificar membresías nulas.

Comparar una columna con None mediante == no identifica
sus nulos. Para ello podemos usar isNull() o eqNullSafe(None).

In [21]:
df_customers.filter(
    col("club_member_status").eqNullSafe("ACTIVE")
).show(3)

print(
    "Membresías nulas:",
    df_customers.filter(
        col("club_member_status").eqNullSafe(None)
    ).count()
)

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|000058a12d5b43e67...|NULL|  NULL|            ACTIVE|                  NONE| 24|64f17e6a330a85798...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows
Membresías nulas: 6062


### Filtrar mediante where()

where() es una alternativa equivalente a filter().
Acepta expresiones de columna y condiciones SQL como texto.

Consultaremos por separado los registros con Active igual
a 1 y los registros con FN igual a 1.

Active y FN son columnas distintas del estado de membresía
club_member_status.


In [22]:
df_customers.where(col("Active") == 1).show()

df_customers.where("FN = 1").show()

+--------------------+---+------+------------------+----------------------+---+--------------------+
|         customer_id| FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+---+------+------------------+----------------------+---+--------------------+
|00006413d8573cd20...|1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
|00007d2de826758b6...|1.0|   1.0|            ACTIVE|             Regularly| 32|8d6f45050876d059c...|
|00009d946eec3ea54...|1.0|   1.0|            ACTIVE|             Regularly| 56|b31984b20a8c478de...|
|0000ae1bbb25e04bd...|1.0|   1.0|            ACTIVE|             Regularly| 29|2c29ae653a9282cce...|
|0000b2f1829e23b24...|1.0|   1.0|            ACTIVE|             Regularly| 54|ca8ca81e8b5794992...|
|0000b95f630aaa931...|1.0|   1.0|            ACTIVE|             Regularly| 49|f63abf76506122d9f...|
|0000d6c053fc8f938...|1.0|   1.0|            ACTIVE|             Regularly| 41|5b5f53c673d0

### Filtrar según la presencia de datos

Realizaremos dos consultas independientes:
- Clientes con club_member_status no nulo.
- Clientes con fashion_news_frequency nulo.

isNotNull() comprueba que el valor esté presente.
isNull() comprueba que sea nulo.

Estas condiciones no detectan cadenas vacías ni espacios.

In [23]:
df_customers.filter(
    col("club_member_status").isNotNull()
).show()

df_customers.filter(
    col("fashion_news_frequency").isNull()
).show()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|000058a12d5b43e67...|NULL|  NULL|            ACTIVE|                  NONE| 24|64f17e6a330a85798...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
|0000757967448a6cb...|NULL|  NULL|            ACTIVE|                  NONE| 20|fe7b8e2b3fafb89ca...|
|00007d2de826758b6...| 1.0|   1.0|            ACTIVE|             Regularly| 32|8d

### Filtrar por contenido de texto

Probaremos tres condiciones sobre postal_code:
- startswith(): comienza con un texto.
- contains(): contiene un texto.
- endswith(): termina con un texto.

En este dataset postal_code está anonimizado.
Estas búsquedas sirven para practicar operaciones de texto,
no para identificar regiones geográficas.

In [24]:
df_customers.filter(
    col("postal_code").startswith("28")
).show()

df_customers.filter(
    col("postal_code").contains("56")
).show()

df_customers.filter(
    col("postal_code").endswith("00")
).show()

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00068c36f7034bf3e...|NULL|  NULL|        PRE-CREATE|                  NONE| 20|28f3300ab2f2a41b2...|
|0016dc6d4cbacb34a...|NULL|  NULL|            ACTIVE|                  NONE| 23|2843f92d2714b40b2...|
|0036620fa407ca305...|NULL|  NULL|            ACTIVE|                  NONE| 22|288e4199ca26dfb67...|
|0037c4ae2ea5c547b...| 1.0|   1.0|            ACTIVE|             Regularly| 21|28f554cac7b568926...|
|00491b4e99e2d274c...| 1.0|   1.0|            ACTIVE|             Regularly| 51|280ec823d1a983daa...|
|004b239de0829da59...|NULL|  NULL|            ACTIVE|                  NONE| 41|285780b21ae65addb...|
|0050b9f0b14dfc5bd...|NULL|  NULL|            ACTIVE|                  NONE| 27|28

### Detectar edades fuera del rango de validación

Identificaremos registros con edades menores que 0
o mayores que 100.

Este rango es una regla de revisión del ejercicio.
Los valores detectados requieren análisis antes de
corregirlos o eliminarlos.

Las edades nulas no cumplen estas comparaciones
y se revisan por separado.


In [25]:
df_edad_invalida = df_customers.filter(
    (col("age") < 0) | (col("age") > 100)
)

df_edad_invalida.count()

0

## 3.6 Ordenar resultados

### Ordenar por edad de forma ascendente

Utilizaremos orderBy("age") para ordenar los registros
por edad, de menor a mayor, y mostraremos tres filas.

Spark coloca los nulos primero en el orden ascendente
predeterminado. Después veremos cómo indicar explícitamente
su posición.


In [26]:
df_customers.orderBy("age").show(3)

+--------------------+----+------+------------------+----------------------+----+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency| age|         postal_code|
+--------------------+----+------+------------------+----------------------+----+--------------------+
|f82ca45bf17bfc11b...|NULL|  NULL|              NULL|                  NULL|NULL|2c29ae653a9282cce...|
|f823998584d60a674...|NULL|  NULL|        PRE-CREATE|                  NONE|NULL|7e8b4d64b2049d53f...|
|f83037644c6ba29ca...|NULL|  NULL|            ACTIVE|                  NONE|NULL|9d7b645592c9c8532...|
+--------------------+----+------+------------------+----------------------+----+--------------------+
only showing top 3 rows


### Ordenar utilizando una expresión de columna

orderBy(col("age")) ordena por edad de menor a mayor,
igual que orderBy("age").

Esta sintaxis permite añadir después opciones como
el sentido del orden o la posición de los nulos.


In [27]:
df_customers.orderBy(col("age")).show(3)


+--------------------+----+------+------------------+----------------------+----+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency| age|         postal_code|
+--------------------+----+------+------------------+----------------------+----+--------------------+
|f82ca45bf17bfc11b...|NULL|  NULL|              NULL|                  NULL|NULL|2c29ae653a9282cce...|
|f823998584d60a674...|NULL|  NULL|        PRE-CREATE|                  NONE|NULL|7e8b4d64b2049d53f...|
|f83037644c6ba29ca...|NULL|  NULL|            ACTIVE|                  NONE|NULL|9d7b645592c9c8532...|
+--------------------+----+------+------------------+----------------------+----+--------------------+
only showing top 3 rows


### Ordenar por edad de forma descendente

Utilizaremos desc() para mostrar primero las edades
más altas.

En el orden descendente predeterminado de Spark,
los valores nulos se colocan al final.


In [28]:
df_customers.orderBy(col("age").desc()).show(3)


+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|7558adbc0401acdd7...|NULL|  NULL|            ACTIVE|                  NONE| 99|f85c74e22df88aabb...|
|687c675e64e5f5d96...|NULL|  NULL|        PRE-CREATE|                  NONE| 99|7d2ef4ec3ff9ea93c...|
|36d219eb822d04d07...|NULL|  NULL|            ACTIVE|                  NONE| 99|f311586e9c31b08f5...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


### Orden descendente mediante ascending=False

Ordenaremos por edad de mayor a menor indicando
ascending=False.

Esta expresión equivale a:
orderBy(col("age").desc()).


In [29]:
df_customers.orderBy("age", ascending=False).show(3)

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|a2106ba216f519880...| 1.0|   1.0|            ACTIVE|             Regularly| 99|cfdea185f4710bf84...|
|a01bd2e0e8bbf8db6...| 1.0|   1.0|            ACTIVE|             Regularly| 99|07d2fb8ade29c6067...|
|a0b9e2473f699821b...|NULL|  NULL|        PRE-CREATE|                  NONE| 99|19aa791d9d42fdad8...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


### Controlar la posición de los nulos

Utilizaremos:
- desc_nulls_last(): orden descendente, con nulos al final.
- asc_nulls_last(): orden ascendente, con nulos al final.

Definirlo explícitamente evita depender del comportamiento
predeterminado del orden.

In [30]:
df_customers.orderBy(
    col("age").desc_nulls_last()
).show(3)

df_customers.orderBy(
    col("postal_code").asc_nulls_last()
).show(3)


+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|e8b2a7bf44f42e808...|NULL|  NULL|            ACTIVE|                  NONE| 99|cc02109b1630ec723...|
|36d219eb822d04d07...|NULL|  NULL|            ACTIVE|                  NONE| 99|f311586e9c31b08f5...|
|3b28156770e0a1743...|NULL|  NULL|            ACTIVE|                  NONE| 99|1cbcd2b9e62968946...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows
+--------------------+---+------+------------------+----------------------+---+--------------------+
|         customer_id| FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+---+------+------------------+--------

### Ordenar por varias columnas

Ordenaremos primero por club_member_status de forma ascendente.

Dentro de cada estado de membresía, ordenaremos age
de mayor a menor.

El segundo criterio se aplica cuando varias filas
comparten el mismo valor del primero.

In [31]:
df_customers.orderBy(
    col("club_member_status").asc(),
    col("age").desc()
).show(3)

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|59027ba9ec753065f...|NULL|  NULL|              NULL|                  NONE| 93|32d5efd0e4ea5baf4...|
|eb65e726a334a1a72...|NULL|  NULL|              NULL|                  NONE| 92|98b4a470dfc22c224...|
|e8bdd5514aaee8211...|NULL|  NULL|              NULL|                  NONE| 86|16794f131214279e0...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


### Ordenar varias columnas mediante listas

Indicaremos las columnas y sus sentidos de orden
en dos listas correspondientes:

- club_member_status: ascendente.
- age: descendente.

Esta expresión equivale al ordenamiento anterior.

In [32]:
df_customers.orderBy(
    ["club_member_status", "age"],
    ascending=[True, False]
).show(3)

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|59027ba9ec753065f...|NULL|  NULL|              NULL|                  NONE| 93|32d5efd0e4ea5baf4...|
|eb65e726a334a1a72...|NULL|  NULL|              NULL|                  NONE| 92|98b4a470dfc22c224...|
|e8bdd5514aaee8211...|NULL|  NULL|              NULL|                  NONE| 86|16794f131214279e0...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 3 rows


### Ordenar por una expresión

Utilizaremos length() para calcular la cantidad de caracteres
de postal_code y ordenar de mayor a menor longitud.

Este ejercicio ordena por una propiedad del texto,
no por el valor del código ni por una ubicación geográfica.

In [33]:
from pyspark.sql.functions import length

df_customers.orderBy(
    length(col("postal_code")).desc()
).show()


+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|f81e0c52aa37179cd...| 1.0|   1.0|            ACTIVE|             Regularly| 27|a780fae71c292f956...|
|f81e12cd4bbdf7e00...|NULL|  NULL|            ACTIVE|                  NONE| 21|85b9c08234d31a072...|
|f81e169a07abceea2...|NULL|  NULL|            ACTIVE|                  NONE| 41|693da7160d65a2850...|
|f81e16c57227a552e...|NULL|  NULL|        PRE-CREATE|                  NONE| 31|059e001dcc7a8f769...|
|f81e1cc287dead341...| 1.0|   1.0|            ACTIVE|             Regularly| 45|9b1c0d6c343f0037e...|
|f81e3a0813e94b4ba...| 1.0|  NULL|            ACTIVE|             Regularly| 32|c91edbea09ba5ab87...|
|f81e3ca07563afd1e...|NULL|  NULL|            ACTIVE|                  NULL| 23|82

### Ordenar mediante sort()

sort() cumple la misma función que orderBy().

Probaremos dos consultas:
- Edad ascendente: de menor a mayor.
- Edad descendente: de mayor a menor.

En el orden predeterminado, los nulos aparecen primero
en ascendente y al final en descendente.


In [34]:
df_customers.sort("age").show(3)

df_customers.sort(col("age").desc()).show(3)

+--------------------+----+------+------------------+----------------------+----+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency| age|         postal_code|
+--------------------+----+------+------------------+----------------------+----+--------------------+
|f82ca45bf17bfc11b...|NULL|  NULL|              NULL|                  NULL|NULL|2c29ae653a9282cce...|
|f823998584d60a674...|NULL|  NULL|        PRE-CREATE|                  NONE|NULL|7e8b4d64b2049d53f...|
|f83037644c6ba29ca...|NULL|  NULL|            ACTIVE|                  NONE|NULL|9d7b645592c9c8532...|
+--------------------+----+------+------------------+----------------------+----+--------------------+
only showing top 3 rows
+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+-----------------

## 3.7 Tratamiento de duplicados

### Identificar clientes con identificador repetido

Agruparemos los registros por customer_id y contaremos
cuántas filas tiene cada identificador.

Mostraremos únicamente los identificadores que aparezcan
más de una vez.

Un identificador repetido no significa necesariamente
que todas las columnas de esas filas sean iguales.

In [35]:
from pyspark.sql.functions import count

df_customers.groupBy(
    "customer_id"
).count().filter(
    "count > 1"
).show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



### Eliminar duplicados de fila completa

Utilizaremos dropDuplicates() sin indicar columnas.

Dos filas se consideran duplicadas cuando coinciden
en todas sus columnas.

Guardaremos el resultado en df_clean y conservaremos
df_customers como DataFrame original.

In [36]:
df_clean = df_customers.dropDuplicates()

### Eliminar duplicados por identificador

Utilizaremos dropDuplicates(["customer_id"]) para conservar
una fila por identificador de cliente.

A diferencia de la deduplicación completa, aquí basta
con que coincida customer_id, aunque otras columnas difieran.

Si hay varias filas del mismo cliente, este método
no garantiza cuál conservará.

In [37]:
df_clean = df_customers.dropDuplicates(["customer_id"])

### Eliminar duplicados por dos columnas

Utilizaremos customer_id y postal_code como clave conjunta.

Una fila se considerará duplicada si coincide en ambas
columnas con otra, aunque los demás campos sean diferentes.

Si un cliente tiene códigos postales distintos,
se conservará una fila por cada combinación.

In [38]:
df_clean = df_customers.dropDuplicates([
    "customer_id",
    "postal_code"
])


### Obtener filas distintas con distinct()

distinct() conserva una sola copia de cada fila completa.

Equivale a dropDuplicates() sin indicar columnas:
la comparación utiliza todos los campos del DataFrame.

Guardaremos esta alternativa en df_clean,
manteniendo df_customers sin cambios.

In [39]:
df_clean = df_customers.distinct()

### Comparar los conteos de deduplicación

Compararemos:
- Total de filas originales.
- Filas distintas considerando todas las columnas.
- Filas conservadas al deduplicar por customer_id.

Si los tres conteos coinciden, no encontramos duplicados
de fila completa ni identificadores repetidos.


In [40]:
total = df_customers.count()

sin_duplicados_fila_completa = df_customers.distinct().count()

sin_duplicados_por_id = df_customers.dropDuplicates(
    ["customer_id"]
).count()

print(
    f"Total: {total}, "
    f"sin duplicar (fila completa): {sin_duplicados_fila_completa}, "
    f"sin duplicar (por customer_id): {sin_duplicados_por_id}"
)

[Stage 70:==>                                                     (1 + 19) / 20]

Total: 1371980, sin duplicar (fila completa): 1371980, sin duplicar (por customer_id): 1371980


### Elegir una fila por cliente mediante una regla

Para cada customer_id, priorizaremos:
1. La edad más alta, colocando los nulos al final.
2. El código postal menor en orden textual como desempate.

Numeraremos las filas de cada cliente y conservaremos
la que tenga el número 1.

Si ambas columnas empatan y otros campos difieren,
esta regla no garantiza una selección única.

In [41]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("customer_id").orderBy(
    col("age").desc_nulls_last(),
    col("postal_code").asc_nulls_last(),
)

df_ranked = df_customers.withColumn(
    "row_num",
    row_number().over(window_spec)
)

df_clean = df_ranked.filter(
    col("row_num") == 1
).drop("row_num")

### Identificar descripciones compartidas por varios artículos

Distintos artículos pueden compartir una descripción,
por ejemplo, cuando son variantes de un producto.

Contaremos las filas por descripción y mostraremos
las cinco descripciones más repetidas.

Excluiremos los nulos: no tener descripción no equivale
a compartir una descripción real.

Esta consulta no elimina artículos.

In [42]:
df_articles = spark.read.csv(
    f"{ORIGEN_DATOS}/articles.csv",
    header=True,
    inferSchema=True
)

duplicados_por_descripcion = (
    df_articles
    .filter(col("detail_desc").isNotNull())
    .groupBy("detail_desc")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)

duplicados_por_descripcion.show(5, truncate=False)

+---------------------------------------------------------------------+-----+
|detail_desc                                                          |count|
+---------------------------------------------------------------------+-----+
|T-shirt in printed cotton jersey.                                    |159  |
|Leggings in soft organic cotton jersey with an elasticated waist.    |138  |
|T-shirt in soft, printed cotton jersey.                              |137  |
|Socks in a soft, jacquard-knit cotton blend with elasticated tops.   |136  |
|Fine-knit trainer socks in a soft cotton blend with elasticated tops.|134  |
+---------------------------------------------------------------------+-----+
only showing top 5 rows


### Seleccionar un representante por producto

product_code identifica el producto base y article_id
identifica sus variantes.

Agruparemos por product_code y conservaremos el artículo
con el menor article_id de cada grupo.

Esta selección reduce variantes para el ejercicio;
no significa que las variantes eliminadas sean errores.
El DataFrame original df_articles se conserva.

In [43]:
window_producto = Window.partitionBy(
    "product_code"
).orderBy("article_id")

df_articles_un_por_producto = (
    df_articles
    .withColumn("fila", row_number().over(window_producto))
    .filter(col("fila") == 1)
    .drop("fila")
)

print(
    f"Filas originales: {df_articles.count()}, "
    f"un representante por product_code: "
    f"{df_articles_un_por_producto.count()}"
)

Filas originales: 105542, un representante por product_code: 47224


## 3.8 Tratar valores nulos

### Rellenar las columnas indicadoras

Siguiendo el criterio de la guía, trataremos los nulos
de FN y Active como indicadores no activados y les
asignaremos el valor 0.

Guardaremos esta prueba en df_fill2.
No rellenaremos la edad con cero, porque eso inventaría
una edad en lugar de representar un dato faltante.


In [44]:
df_fill2 = df_customers.fillna({
    "FN": 0,
    "Active": 0
})


### Aplicar el tratamiento de nulos por columna

Crearemos df_customers_limpio con estas reglas:
- FN y Active: reemplazar nulos por 0.
- fashion_news_frequency: reemplazar nulos por NONE.
- club_member_status: reemplazar nulos por UNKNOWN.

Mantendremos los nulos de age para no inventar edades.
Estas reglas corresponden al criterio adoptado en la guía.

In [45]:
df_customers_limpio = df_customers.na.fill({
    "FN": 0,
    "Active": 0,
    "fashion_news_frequency": "NONE",
    "club_member_status": "UNKNOWN",
})


### Evaluar el efecto de eliminar cualquier fila con nulos

Aplicaremos na.drop() al DataFrame original y contaremos
las filas restantes.

Sin indicar columnas, na.drop() descarta una fila
si cualquiera de sus campos es nulo.

Esta consulta es una prueba; no será nuestra versión final.

In [46]:
df_customers.na.drop().count()

462911

### Excluir registros sin identificador

customer_id es necesario para identificar a cada cliente.

Aplicaremos na.drop() únicamente sobre esa columna,
utilizando el DataFrame con los nulos ya tratados.

Esta operación elimina identificadores nulos,
pero no cadenas vacías ni espacios, que revisamos antes.

In [47]:
df_customers_valido = df_customers_limpio.na.drop(
    subset=["customer_id"]
)

print(
    f"Filas antes: {df_customers.count()}, "
    f"después de na.drop(subset=['customer_id']): "
    f"{df_customers_valido.count()}"
)

Filas antes: 1371980, después de na.drop(subset=['customer_id']): 1371980


### Confirmar que no quedan identificadores nulos

Comprobaremos que df_customers_valido no contiene
valores nulos en customer_id.

Utilizaremos assert para señalar un incumplimiento
de esta condición antes de continuar.

Esta comprobación no detecta identificadores vacíos
ni repetidos; esos controles se realizaron por separado.

In [48]:
assert df_customers_valido.filter(
    col("customer_id").isNull()
).count() == 0

### Reutilizar el DataFrame mediante cache()

Utilizaremos df_customers_valido en varias operaciones.

cache() permite conservar sus particiones después de
calcularlas, evitando repetir la lectura y la limpieza
en cada consulta.

El almacenamiento se materializa cuando una acción
solicita los datos, no al escribir cache().

In [49]:
df_customers_valido = df_customers_valido.cache()

### Elegir el almacenamiento mediante persist()

Utilizaremos MEMORY_AND_DISK para conservar las particiones
en memoria y permitir su almacenamiento en disco cuando
no quepan en ella.

Primero retiraremos la configuración de caché anterior
con unpersist(). Esto no elimina el DataFrame ni sus archivos.

La próxima acción materializará las particiones.


In [50]:
from pyspark.storagelevel import StorageLevel

df_customers_valido.unpersist()

df_customers_valido = df_customers_valido.persist(
    StorageLevel.MEMORY_AND_DISK
)

## 3.9 Escritura en múltiples formatos

### Preparar una muestra

Utilizaremos hasta 100 registros de df_edad_invalida,
creado durante la validación de edades.

Si no encontramos edades fuera del rango, la muestra
estará vacía. Eso es válido para este ejercicio.

Después guardaremos la muestra en CSV, JSON y Parquet.

In [51]:
muestra = df_edad_invalida.limit(100)

### Escribir la muestra en tres formatos

Guardaremos la misma muestra en CSV, JSON y Parquet.

Cada salida se escribirá en una carpeta independiente
dentro de artifacts.

El modo overwrite reemplaza esas carpetas de salida
si ya existen. No modifica los archivos originales.

In [52]:
muestra.write.format("csv").option(
    "header", True
).mode("overwrite").save(
    f"{ARTIFACTS}/muestra_csv"
)

muestra.write.format("json").mode("overwrite").save(
    f"{ARTIFACTS}/muestra_json"
)

muestra.write.format("parquet").mode("overwrite").save(
    f"{ARTIFACTS}/muestra_parquet"
)

## 3.10 Escritura particionada en Parquet

Guardaremos df_customers_valido en Parquet, organizando
las carpetas según club_member_status.

Esta columna tiene pocas categorías y puede utilizarse
para filtrar consultas.

repartition(4) redistribuirá los datos en cuatro particiones
de procesamiento antes de escribir.

Esto no garantiza cuatro archivos por cada carpeta.

In [53]:
(
    df_customers_valido
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("club_member_status")
    .save(f"{ARTIFACTS}/customers_particionado")
)

### Verificar las carpetas de partición

Listaremos el contenido de customers_particionado.

Esperamos encontrar carpetas con nombres como:
club_member_status=ACTIVE.

Cada carpeta agrupa los registros de una categoría.
También pueden aparecer archivos de control, como _SUCCESS.

In [54]:
import os

for carpeta in sorted(
    os.listdir(f"{ARTIFACTS}/customers_particionado")
):
    print(carpeta)

._SUCCESS.crc
_SUCCESS
club_member_status=ACTIVE
club_member_status=LEFT CLUB
club_member_status=PRE-CREATE
club_member_status=UNKNOWN


### Comparar la escritura con coalesce(1)

Reduciremos el procesamiento a una sola partición
antes de escribir.

Mantendremos las carpetas por club_member_status,
pero en este ejercicio esperamos un archivo de datos
por cada categoría.

Esta opción reduce el paralelismo de escritura
y no resulta adecuada para cualquier volumen de datos.

In [55]:
(
    df_customers_valido
    .coalesce(1)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("club_member_status")
    .save(f"{ARTIFACTS}/customers_particionado_un_archivo")
)

## 3.11 Leer y verificar la salida particionada

Leeremos la carpeta customers_particionado como un DataFrame.

Revisaremos su esquema y contaremos los registros.
Spark recuperará club_member_status a partir de los nombres
de las carpetas de partición.

Después compararemos el conteo con el DataFrame guardado.


In [56]:
df_verificacion = spark.read.parquet(
    f"{ARTIFACTS}/customers_particionado"
)

df_verificacion.printSchema()

df_verificacion.count()

root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: double (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- club_member_status: string (nullable = true)



1371980

### Comparar el conteo de ida y vuelta

Verificaremos que la salida Parquet recuperada tenga
la misma cantidad de registros que df_customers_valido.

Esta comprobación confirma la igualdad de los conteos,
pero no demuestra por sí sola que todos los valores
sean idénticos.

In [57]:
assert df_verificacion.count() == df_customers_valido.count()

### Revisar el filtro sobre la columna particionada

Filtraremos club_member_status por ACTIVE y consultaremos
el plan de ejecución.

Buscaremos PartitionFilters en la lectura Parquet.
Una condición en ese apartado indica que Spark puede
seleccionar particiones antes de leer sus archivos de datos.

In [58]:
df_verificacion.filter(
    col("club_member_status") == "ACTIVE"
).explain(True)

== Parsed Logical Plan ==
'Filter '`=`('club_member_status, ACTIVE)
+- Relation [customer_id#1673,FN#1674,Active#1675,fashion_news_frequency#1676,age#1677,postal_code#1678,club_member_status#1679] parquet

== Analyzed Logical Plan ==
customer_id: string, FN: double, Active: double, fashion_news_frequency: string, age: int, postal_code: string, club_member_status: string
Filter (club_member_status#1679 = ACTIVE)
+- Relation [customer_id#1673,FN#1674,Active#1675,fashion_news_frequency#1676,age#1677,postal_code#1678,club_member_status#1679] parquet

== Optimized Logical Plan ==
Filter (isnotnull(club_member_status#1679) AND (club_member_status#1679 = ACTIVE))
+- Relation [customer_id#1673,FN#1674,Active#1675,fashion_news_frequency#1676,age#1677,postal_code#1678,club_member_status#1679] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [customer_id#1673,FN#1674,Active#1675,fashion_news_frequency#1676,age#1677,postal_code#1678,club_member_status#1679] Batched: true, DataFi

### Comparar el filtro utilizando upper()

upper() convierte el texto a mayúsculas antes de compararlo
con ACTIVE.

Revisaremos el plan para comprobar si Spark mantiene
un filtro de particiones con esta expresión.

El comportamiento debe verificarse en la salida real:
no asumiremos que upper() impide la poda de particiones.

In [59]:
from pyspark.sql.functions import upper

df_verificacion.filter(
    upper(col("club_member_status")) == "ACTIVE"
).explain(True)

== Parsed Logical Plan ==
'Filter '`=`('upper('club_member_status), ACTIVE)
+- Relation [customer_id#1673,FN#1674,Active#1675,fashion_news_frequency#1676,age#1677,postal_code#1678,club_member_status#1679] parquet

== Analyzed Logical Plan ==
customer_id: string, FN: double, Active: double, fashion_news_frequency: string, age: int, postal_code: string, club_member_status: string
Filter (upper(club_member_status#1679) = ACTIVE)
+- Relation [customer_id#1673,FN#1674,Active#1675,fashion_news_frequency#1676,age#1677,postal_code#1678,club_member_status#1679] parquet

== Optimized Logical Plan ==
Filter (isnotnull(club_member_status#1679) AND (upper(club_member_status#1679) = ACTIVE))
+- Relation [customer_id#1673,FN#1674,Active#1675,fashion_news_frequency#1676,age#1677,postal_code#1678,club_member_status#1679] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [customer_id#1673,FN#1674,Active#1675,fashion_news_frequency#1676,age#1677,postal_code#1678,club_member_status#1679]

### Contar registros por partición

Agruparemos los datos recuperados por club_member_status
y contaremos las filas de cada categoría.

Esto permite observar cómo se distribuyen los registros
entre las carpetas de partición y detectar desequilibrios.

In [60]:
df_verificacion.groupBy(
    "club_member_status"
).count().orderBy(
    "club_member_status"
).show(truncate=False)

+------------------+-------+
|club_member_status|count  |
+------------------+-------+
|ACTIVE            |1272491|
|LEFT CLUB         |467    |
|PRE-CREATE        |92960  |
|UNKNOWN           |6062   |
+------------------+-------+



### Liberar el almacenamiento persistido

Ya terminamos de reutilizar df_customers_valido.

Utilizaremos unpersist() para liberar sus particiones
almacenadas en memoria o disco.

Esto no elimina el DataFrame ni los archivos Parquet.
Si volvemos a solicitar sus datos, Spark podrá recalcularlos.

In [61]:
df_customers_valido.unpersist()

DataFrame[customer_id: string, FN: double, Active: double, club_member_status: string, fashion_news_frequency: string, age: int, postal_code: string]

## 3.12 Hallazgos de la práctica guiada

### Esquema y calidad de datos

Cargué customers.csv con un esquema explícito de siete columnas.
Revisé los nombres, tipos y cantidad de registros.

Cuantifiqué los valores nulos por columna y comprobé
identificadores nulos, vacíos o formados por espacios.

### Tratamiento aplicado

Siguiendo el criterio de la guía:
- Rellené FN y Active con 0.
- Rellené fashion_news_frequency con NONE.
- Rellené club_member_status con UNKNOWN.
- Conservé las edades nulas para no inventar valores.
- Excluí registros con customer_id nulo.

### Duplicados

Comparé la deduplicación de filas completas y por customer_id.
También utilicé una ventana para seleccionar un representante
por product_code, sin considerar erróneas sus variantes.

### Escritura y lectura

Guardé los clientes tratados en Parquet particionado
por club_member_status y leí nuevamente la salida.

El assert de comparación de conteos terminó: sin error.

### Resultados de mi ejecución

- Filas originales de clientes: 1 371 980.
- Filas distintas completas: 1 371 980.
- Filas distintas por customer_id: 1 371 980.
- Filas de clientes después de la limpieza: 1 371 980.
- Artículos originales: 105 542.
- Representantes por product_code: 47 224.

In [62]:
print("FILTRO DIRECTO")
df_verificacion.filter(
    col("club_member_status") == "ACTIVE"
).explain()

print("FILTRO CON UPPER")
df_verificacion.filter(
    upper(col("club_member_status")) == "ACTIVE"
).explain()

FILTRO DIRECTO
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [customer_id#1673,FN#1674,Active#1675,fashion_news_frequency#1676,age#1677,postal_code#1678,club_member_status#1679] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/s03-procesamiento-calidad-datos/artifacts/customers_particio..., PartitionFilters: [isnotnull(club_member_status#1679), (club_member_status#1679 = ACTIVE)], PushedFilters: [], ReadSchema: struct<customer_id:string,FN:double,Active:double,fashion_news_frequency:string,age:int,postal_co...


FILTRO CON UPPER
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [customer_id#1673,FN#1674,Active#1675,fashion_news_frequency#1676,age#1677,postal_code#1678,club_member_status#1679] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/s03-procesamiento-calidad-datos/artifacts/customers_particio..., PartitionFilters: [isnotnull(club_member_status#1679), (upper(club_m

## 4. Actividad autónoma: calidad de datos en AgroSmart Puno

### Objetivo

Aplicar los controles de calidad de S03 a los datos
agroclimáticos utilizados en S02.

### Datos de trabajo

Registros horarios de Open-Meteo para un punto de referencia
de Azángaro, correspondientes a junio de 2025.

### Controles previstos

- Validar el esquema y las columnas obligatorias.
- Detectar nulos, identificadores vacíos y duplicados.
- Filtrar y ordenar mediante distintas técnicas.
- Documentar las decisiones de tratamiento.
- Guardar una salida Parquet particionada.
- Leerla nuevamente y comprobar conteos y PartitionFilters.

### Alcance

Los datos proceden de modelos meteorológicos.
Los controles de calidad no los convierten en mediciones
de sensores ni confirman daños en cultivos.

In [63]:
from pathlib import Path

carpeta_originales = Path(
    "/opt/s02-fundamentos/data/agrosmart"
)

archivos_json = sorted(
    carpeta_originales.glob("open_meteo_azangaro_*.json")
)

print("Archivos encontrados:", len(archivos_json))

for archivo in archivos_json:
    print(archivo.name)

Archivos encontrados: 1
open_meteo_azangaro_20261116T210745703464Z.json


### Abrir y comprobar la descarga original

Leeremos una respuesta JSON conservada en S02.

Revisaremos la fuente, la fecha de descarga, los parámetros
y el periodo de sus registros antes de utilizarla en S03.

Este paso no modifica el archivo original.

In [64]:
import json

if not archivos_json:
    raise FileNotFoundError(
        "No se encontraron los JSON de AgroSmart en S02."
    )

ruta_origen_agro = archivos_json[-1]

with ruta_origen_agro.open("r", encoding="utf-8") as archivo:
    descarga_agro = json.load(archivo)

respuesta_agro = descarga_agro["respuesta"]
horarios_agro = respuesta_agro["hourly"]

print("Archivo:", ruta_origen_agro.name)
print("Fuente:", descarga_agro.get("fuente"))
print("Fecha de descarga UTC:", descarga_agro.get("fecha_descarga_utc"))
print("Parámetros:", descarga_agro.get("parametros"))
print("Primer registro:", horarios_agro["time"][0])
print("Último registro:", horarios_agro["time"][-1])
print("Cantidad de horas:", len(horarios_agro["time"]))

Archivo: open_meteo_azangaro_20261116T210745703464Z.json
Fuente: Open-Meteo Historical Weather API
Fecha de descarga UTC: 2026-11-16T21:07:45.703464+00:00
Parámetros: {'latitude': -14.91, 'longitude': -70.2, 'start_date': '2025-06-01', 'end_date': '2025-06-30', 'hourly': 'temperature_2m,relative_humidity_2m,soil_temperature_0_to_7cm,soil_moisture_0_to_7cm', 'timezone': 'America/Lima'}
Primer registro: 2025-06-01T00:00
Último registro: 2025-06-30T23:00
Cantidad de horas: 720


### Conservar la entrada Bronze de AgroSmart

Copiaremos la descarga original de S02 a data/agrosmart/bronze
dentro de S03.

Bronze conserva los datos recibidos sin aplicar limpieza.
Mantendremos también el archivo original de S02.

In [65]:
carpeta_bronze = Path(ORIGEN_DATOS) / "agrosmart" / "bronze"
carpeta_bronze.mkdir(parents=True, exist_ok=True)

ruta_bronze = carpeta_bronze / ruta_origen_agro.name
contenido_original = ruta_origen_agro.read_bytes()

if ruta_bronze.exists():
    if ruta_bronze.read_bytes() != contenido_original:
        raise ValueError(
            "Ya existe una copia con contenido diferente."
        )
    print("La copia ya existe y coincide con el original.")
else:
    with ruta_bronze.open("xb") as archivo:
        archivo.write(contenido_original)
    print("Copia Bronze creada.")

assert ruta_bronze.read_bytes() == contenido_original

print("Archivo verificado:", ruta_bronze)

Copia Bronze creada.
Archivo verificado: /opt/s03-procesamiento-calidad-datos/data/agrosmart/bronze/open_meteo_azangaro_20261116T210745703464Z.json


### Validar la estructura de los datos originales

Comprobaremos que existan las cinco listas necesarias:
fecha y hora, temperatura del aire, humedad relativa,
temperatura del suelo y humedad del suelo.

Todas deben tener la misma cantidad de elementos,
porque cada posición representa una misma hora.

Si falta una variable o las longitudes difieren,
detendremos el procesamiento.

variables_requeridas = {
    "time",
    "temperature_2m",
    "relative_humidity_2m",
    "soil_temperature_0_to_7cm",
    "soil_moisture_0_to_7cm",
}

faltantes = variables_requeridas - set(horarios_agro)

if faltantes:
    raise ValueError(f"Faltan variables: {sorted(faltantes)}")

for variable in sorted(variables_requeridas):
    if not isinstance(horarios_agro[variable], list):
        raise TypeError(f"{variable} debe ser una lista.")

longitudes = {
    variable: len(horarios_agro[variable])
    for variable in sorted(variables_requeridas)
}

if len(set(longitudes.values())) != 1:
    raise ValueError(f"Las longitudes no coinciden: {longitudes}")

if longitudes["time"] == 0:
    raise ValueError("La respuesta no contiene registros.")

print("Estructura validada.")
print("Cantidad de elementos por variable:", longitudes)

### Preparar los registros horarios

Uniremos las variables que ocupan la misma posición
en las listas de la API.

Cada registro tendrá fecha y hora, localidad y las cuatro
variables ambientales. Conservaremos los valores nulos
sin reemplazarlos por cero.

La localidad identifica el punto de referencia consultado,
no una parcela con sensores.

In [67]:
registros_agro_s03 = []

for i, fecha_hora in enumerate(horarios_agro["time"]):
    registros_agro_s03.append({
        "fecha_hora": fecha_hora,
        "localidad": "Azangaro",
        "temperatura_aire_c": horarios_agro["temperature_2m"][i],
        "humedad_relativa_pct": horarios_agro["relative_humidity_2m"][i],
        "temperatura_suelo_c": horarios_agro["soil_temperature_0_to_7cm"][i],
        "humedad_suelo_m3_m3": horarios_agro["soil_moisture_0_to_7cm"][i],
    })

print("Registros preparados:", len(registros_agro_s03))
print("Primer registro:", registros_agro_s03[0])

Registros preparados: 720
Primer registro: {'fecha_hora': '2025-06-01T00:00', 'localidad': 'Azangaro', 'temperatura_aire_c': 2.9, 'humedad_relativa_pct': 70, 'temperatura_suelo_c': 4.3, 'humedad_suelo_m3_m3': 0.212}


### Definir el esquema de AgroSmart

Estableceremos los nombres y tipos esperados:
- fecha_hora y localidad: texto.
- Las cuatro variables ambientales: números decimales.

Permitiremos nulos inicialmente para detectarlos
y decidir su tratamiento de forma explícita.

Definir el esquema no demuestra que los valores sean válidos.

In [68]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

schema_agro_s03 = StructType([
    StructField("fecha_hora", StringType(), True),
    StructField("localidad", StringType(), True),
    StructField("temperatura_aire_c", DoubleType(), True),
    StructField("humedad_relativa_pct", DoubleType(), True),
    StructField("temperatura_suelo_c", DoubleType(), True),
    StructField("humedad_suelo_m3_m3", DoubleType(), True),
])

### Crear y explorar el DataFrame

Convertiremos los valores ambientales disponibles a números
decimales, conservando los nulos.

Crearemos df_agro_s03 con el esquema explícito y revisaremos
sus columnas, cantidad de registros y primeras filas.

Esta preparación no modifica el JSON original.

In [69]:
columnas_numericas = {
    "temperatura_aire_c",
    "humedad_relativa_pct",
    "temperatura_suelo_c",
    "humedad_suelo_m3_m3",
}

registros_para_spark = [
    {
        nombre: (
            float(valor)
            if nombre in columnas_numericas and valor is not None
            else valor
        )
        for nombre, valor in registro.items()
    }
    for registro in registros_agro_s03
]

df_agro_s03 = spark.createDataFrame(
    registros_para_spark,
    schema=schema_agro_s03
)

columnas_esperadas = set(schema_agro_s03.fieldNames())
assert set(df_agro_s03.columns) == columnas_esperadas

df_agro_s03.printSchema()
print("Total de registros:", df_agro_s03.count())
df_agro_s03.show(5, truncate=False)

root
 |-- fecha_hora: string (nullable = true)
 |-- localidad: string (nullable = true)
 |-- temperatura_aire_c: double (nullable = true)
 |-- humedad_relativa_pct: double (nullable = true)
 |-- temperatura_suelo_c: double (nullable = true)
 |-- humedad_suelo_m3_m3: double (nullable = true)



[Stage 126:>                                                      (0 + 20) / 20]

Total de registros: 720
+----------------+---------+------------------+--------------------+-------------------+-------------------+
|fecha_hora      |localidad|temperatura_aire_c|humedad_relativa_pct|temperatura_suelo_c|humedad_suelo_m3_m3|
+----------------+---------+------------------+--------------------+-------------------+-------------------+
|2025-06-01T00:00|Azangaro |2.9               |70.0                |4.3                |0.212              |
|2025-06-01T01:00|Azangaro |2.9               |70.0                |4.0                |0.212              |
|2025-06-01T02:00|Azangaro |2.8               |70.0                |3.8                |0.212              |
|2025-06-01T03:00|Azangaro |2.1               |74.0                |3.5                |0.212              |
|2025-06-01T04:00|Azangaro |1.9               |76.0                |3.5                |0.212              |
+----------------+---------+------------------+--------------------+-------------------+----------------

### Diagnóstico de calidad

Revisaremos:
- Valores nulos en todas las columnas.
- Fechas y localidades vacías o formadas por espacios.
- Registros repetidos para una misma localidad y hora.

La clave propuesta es localidad + fecha_hora.
Primero identificaremos los problemas y después
decidiremos su tratamiento.

In [70]:
from pyspark.sql import functions as F

# Nulos por columna
df_agro_s03.select([
    F.count(
        F.when(F.col(nombre).isNull(), 1)
    ).alias(f"nulos_{nombre}")
    for nombre in df_agro_s03.columns
]).show(vertical=True)

# Textos vacíos o compuestos por espacios
df_agro_s03.select([
    F.count(
        F.when(F.trim(F.col(nombre)) == "", 1)
    ).alias(f"vacios_{nombre}")
    for nombre in ["fecha_hora", "localidad"]
]).show()

# Claves repetidas
duplicados_agro = (
    df_agro_s03
    .groupBy("localidad", "fecha_hora")
    .count()
    .filter(F.col("count") > 1)
)

duplicados_agro.show(truncate=False)

-RECORD 0-------------------------
 nulos_fecha_hora           | 0   
 nulos_localidad            | 0   
 nulos_temperatura_aire_c   | 0   
 nulos_humedad_relativa_pct | 0   
 nulos_temperatura_suelo_c  | 0   
 nulos_humedad_suelo_m3_m3  | 0   

+-----------------+----------------+
|vacios_fecha_hora|vacios_localidad|
+-----------------+----------------+
|                0|               0|
+-----------------+----------------+

+---------+----------+-----+
|localidad|fecha_hora|count|
+---------+----------+-----+
+---------+----------+-----+



### Validar fechas y rangos

Convertiremos fecha_hora a timestamp en America/Lima.

Identificaremos:
- Fechas nulas o que no puedan convertirse.
- Humedad relativa fuera de 0 a 100 %.
- Humedad volumétrica del suelo fuera de 0 a 1 m³/m³.

Estos controles detectan problemas básicos; no garantizan
la exactitud de las mediciones. No consideraremos inválida
una temperatura solo por ser negativa.

In [71]:
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

df_agro_revision = df_agro_s03.withColumn(
    "fecha_hora_ts",
    F.try_to_timestamp(
        F.col("fecha_hora"),
        F.lit("yyyy-MM-dd'T'HH:mm")
    )
)

df_agro_revision.select(
    F.count(
        F.when(F.col("fecha_hora_ts").isNull(), 1)
    ).alias("fechas_nulas_o_no_convertibles"),

    F.count(
        F.when(
            ~F.col("humedad_relativa_pct").between(0, 100),
            1
        )
    ).alias("humedad_aire_fuera_de_rango"),

    F.count(
        F.when(
            ~F.col("humedad_suelo_m3_m3").between(0, 1),
            1
        )
    ).alias("humedad_suelo_fuera_de_rango")
).show(vertical=True)

-RECORD 0-----------------------------
 fechas_nulas_o_no_convertibles | 0   
 humedad_aire_fuera_de_rango    | 0   
 humedad_suelo_fuera_de_rango   | 0   



### Preparar datos válidos y registros para revisión

Normalizaremos los espacios de localidad y exigiremos:
- Localidad no nula ni vacía.
- Fecha y hora convertibles.
- Humedades dentro de sus rangos cuando estén disponibles.

Conservaremos los nulos ambientales sin inventar valores.
Separaremos los registros rechazados para revisarlos.

Eliminaremos únicamente filas completamente idénticas.
Si una misma localidad y hora tiene valores diferentes,
el procesamiento se detendrá para revisar el conflicto.

In [72]:
df_agro_normalizado = df_agro_revision.withColumn(
    "localidad",
    F.trim(F.col("localidad"))
)

condicion_valida = (
    F.col("localidad").isNotNull()
    & (F.col("localidad") != "")
    & F.col("fecha_hora_ts").isNotNull()
    & (
        F.col("humedad_relativa_pct").isNull()
        | F.col("humedad_relativa_pct").between(0, 100)
    )
    & (
        F.col("humedad_suelo_m3_m3").isNull()
        | F.col("humedad_suelo_m3_m3").between(0, 1)
    )
)

df_agro_rechazados = df_agro_normalizado.filter(
    ~condicion_valida
)

df_agro_silver = (
    df_agro_normalizado
    .filter(condicion_valida)
    .dropDuplicates()
)

conflictos = (
    df_agro_silver
    .groupBy("localidad", "fecha_hora_ts")
    .count()
    .filter(F.col("count") > 1)
)

print("Registros originales:", df_agro_s03.count())
print("Registros para revisión:", df_agro_rechazados.count())
print("Registros Silver:", df_agro_silver.count())

if conflictos.count() > 0:
    conflictos.show(truncate=False)
    raise ValueError(
        "Hay varias filas diferentes para una localidad y hora."
    )

Registros originales: 720
Registros para revisión: 0
Registros Silver: 720


### Filtrar y ordenar los datos Silver

Consultaremos temperaturas del aire a cero o bajo cero
mediante dos formas equivalentes:
- filter() con una expresión de columna.
- where() con una condición SQL.

Ordenaremos los resultados utilizando orderBy() y sort().
Estos ejemplos no modifican df_agro_silver.

In [73]:
# Forma 1: filter() y orderBy()
df_agro_silver.filter(
    F.col("temperatura_aire_c") <= 0
).orderBy(
    F.col("fecha_hora_ts").asc()
).show(5, truncate=False)

# Forma 2: where() y sort()
df_agro_silver.where(
    "temperatura_aire_c <= 0"
).sort(
    F.col("temperatura_aire_c").asc(),
    F.col("fecha_hora_ts").asc()
).show(5, truncate=False)

+----------------+---------+------------------+--------------------+-------------------+-------------------+-------------------+
|fecha_hora      |localidad|temperatura_aire_c|humedad_relativa_pct|temperatura_suelo_c|humedad_suelo_m3_m3|fecha_hora_ts      |
+----------------+---------+------------------+--------------------+-------------------+-------------------+-------------------+
|2025-06-02T05:00|Azangaro |-0.2              |94.0                |3.1                |0.208              |2025-06-02 05:00:00|
|2025-06-02T06:00|Azangaro |-0.4              |99.0                |2.9                |0.208              |2025-06-02 06:00:00|
|2025-06-03T03:00|Azangaro |-0.1              |68.0                |2.2                |0.197              |2025-06-03 03:00:00|
|2025-06-03T04:00|Azangaro |-0.6              |71.0                |1.8                |0.197              |2025-06-03 04:00:00|
|2025-06-03T05:00|Azangaro |-1.4              |76.0                |1.8                |0.197    

### Preparar el particionamiento temporal

Crearemos las columnas anio y mes a partir de fecha_hora_ts.

Organizaremos la salida Parquet por año y mes, evitando
crear una carpeta por cada registro horario.

En esta muestra solo esperamos junio de 2025.
El particionamiento demuestra la estructura; con un único
mes, su beneficio para reducir lecturas será limitado.

In [74]:
df_agro_silver_particionado = (
    df_agro_silver
    .withColumn(
        "anio",
        F.year(F.col("fecha_hora_ts"))
    )
    .withColumn(
        "mes",
        F.month(F.col("fecha_hora_ts"))
    )
)

df_agro_silver_particionado.select(
    "fecha_hora_ts",
    "anio",
    "mes"
).show(5, truncate=False)

+-------------------+----+---+
|fecha_hora_ts      |anio|mes|
+-------------------+----+---+
|2025-06-01 04:00:00|2025|6  |
|2025-06-01 11:00:00|2025|6  |
|2025-06-01 17:00:00|2025|6  |
|2025-06-02 06:00:00|2025|6  |
|2025-06-01 07:00:00|2025|6  |
+-------------------+----+---+
only showing top 5 rows


### Guardar y verificar el Parquet particionado

Escribiremos los datos Silver en carpetas por año y mes.

Después leeremos la salida y compararemos su cantidad
de registros con el DataFrame anterior a la escritura.

La igualdad de conteos es un control básico; no demuestra
por sí sola que todos los valores sean idénticos.

In [75]:
RUTA_SILVER_AGRO = f"{ARTIFACTS}/agrosmart/silver"

(
    df_agro_silver_particionado
    .write
    .mode("overwrite")
    .partitionBy("anio", "mes")
    .parquet(RUTA_SILVER_AGRO)
)

df_agro_verificacion = spark.read.parquet(RUTA_SILVER_AGRO)

filas_antes = df_agro_silver_particionado.count()
filas_despues = df_agro_verificacion.count()

print("Filas antes de guardar:", filas_antes)
print("Filas después de leer:", filas_despues)

assert filas_antes == filas_despues, (
    "Los conteos no coinciden."
)

df_agro_verificacion.printSchema()

Filas antes de guardar: 720
Filas después de leer: 720
root
 |-- fecha_hora: string (nullable = true)
 |-- localidad: string (nullable = true)
 |-- temperatura_aire_c: double (nullable = true)
 |-- humedad_relativa_pct: double (nullable = true)
 |-- temperatura_suelo_c: double (nullable = true)
 |-- humedad_suelo_m3_m3: double (nullable = true)
 |-- fecha_hora_ts: timestamp (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)



### Verificar PartitionFilters en AgroSmart

Consultaremos los registros de junio de 2025 utilizando
las columnas de partición anio y mes.

Revisaremos PartitionFilters en el plan físico para comprobar
que Spark reconoce las condiciones sobre las particiones.

Como la muestra solo contiene ese mes, el filtro no permite
demostrar una reducción efectiva de archivos leídos.


In [76]:
df_agro_junio = df_agro_verificacion.filter(
    (F.col("anio") == 2025)
    & (F.col("mes") == 6)
)

df_agro_junio.explain(True)

== Parsed Logical Plan ==
'Filter 'and('`=`('anio, 2025), '`=`('mes, 6))
+- Relation [fecha_hora#2161,localidad#2162,temperatura_aire_c#2163,humedad_relativa_pct#2164,temperatura_suelo_c#2165,humedad_suelo_m3_m3#2166,fecha_hora_ts#2167,anio#2168,mes#2169] parquet

== Analyzed Logical Plan ==
fecha_hora: string, localidad: string, temperatura_aire_c: double, humedad_relativa_pct: double, temperatura_suelo_c: double, humedad_suelo_m3_m3: double, fecha_hora_ts: timestamp, anio: int, mes: int
Filter ((anio#2168 = 2025) AND (mes#2169 = 6))
+- Relation [fecha_hora#2161,localidad#2162,temperatura_aire_c#2163,humedad_relativa_pct#2164,temperatura_suelo_c#2165,humedad_suelo_m3_m3#2166,fecha_hora_ts#2167,anio#2168,mes#2169] parquet

== Optimized Logical Plan ==
Filter ((isnotnull(anio#2168) AND isnotnull(mes#2169)) AND ((anio#2168 = 2025) AND (mes#2169 = 6)))
+- Relation [fecha_hora#2161,localidad#2162,temperatura_aire_c#2163,humedad_relativa_pct#2164,temperatura_suelo_c#2165,humedad_suelo_m3_m3

### Resultados de calidad de AgroSmart Puno

#### Fuente y alcance
Utilicé la descarga histórica de Open-Meteo conservada
en S02 para Azángaro, correspondiente a junio de 2025.
La copia Bronze mantiene el contenido original.

#### Controles aplicados
- Esquema explícito y columnas requeridas.
- Nulos y textos vacíos.
- Conversión de fecha y hora.
- Rangos de humedad ambiental y del suelo.
- Duplicados completos y conflictos por localidad y hora.

#### Tratamiento
Conservé los nulos ambientales sin sustituirlos por cero.
Separé los registros que incumplían las reglas.
Eliminé únicamente filas completamente idénticas.
Los conflictos por localidad y hora detienen el proceso.

#### Resultados observados
- Registros originales: 720.
- Registros para revisión: 0.
- Registros Silver: 720.
- Registros recuperados del Parquet: 720.
- Comparación de conteos: el assert se cumplió sin error.

#### Salida analítica
Guardé Silver en Parquet particionado por año y mes.
La muestra contiene un solo mes, por lo que no permite
demostrar ahorro de lectura entre varios periodos.

#### Límites
Estos controles no garantizan exactitud meteorológica.
Los datos del modelo no equivalen a sensores de una parcela
ni confirman daños en cultivos.

### Hallazgo técnico: repetición no equivale a error

#### Qué observé
Al agrupar articles.csv por detail_desc, encontré descripciones
compartidas por varios registros.

#### Cómo lo identifiqué
Filtré descripciones no nulas, agrupé por detail_desc
y consulté los grupos con más de una fila.

#### Decisión tomada
No eliminé artículos solo por compartir una descripción,
porque pueden representar variantes de un producto.

Para obtener un representante por product_code utilicé
Window y row_number(), ordenando por article_id.

#### Aplicación a AgroSmart
Dos registros con la misma temperatura no son necesariamente
duplicados. Revisé la clave localidad y fecha_hora_ts.

Si esa clave aparece con valores diferentes, el proceso
se detiene para analizar el conflicto antes de elegir una fila.

### Reflexión técnica

dropDuplicates() elimina repeticiones según las columnas indicadas, sin priorizar una fila.  
Window y row_number() permiten seleccionar un representante con una regla de orden.  
En articles.csv, compartir una descripción no demuestra que dos artículos sean duplicados.  
Siguiendo la guía, rellené FN y Active con 0 y las categorías ausentes con NONE o UNKNOWN.  
Conservé las edades nulas, porque reemplazarlas por cero inventaría información.  
between(25, 35) incluye ambos límites; las comparaciones estrictas los excluyen.  
PushedFilters entrega condiciones al lector; PartitionFilters selecciona particiones.

### Interpretación de los planes de particionamiento

En H&M, tanto el filtro directo por club_member_status como el filtro con upper() presentan condiciones dentro de PartitionFilters. En esta ejecución, upper() no eliminó el filtro de particiones. Esto difiere del comportamiento anticipado por la guía; se documenta el plan realmente obtenido, sin afirmar una pérdida de poda que no aparece en él.

En AgroSmart, PartitionFilters contiene condiciones de anio = 2025 y mes = 6, junto con las comprobaciones de no nulos. Spark reconoce las columnas de partición. Como la muestra solo contiene junio de 2025, este filtro no demuestra ahorro de lectura frente a otros meses.

En estas lecturas PushedFilters aparece vacío: las condiciones consultadas se aplican a columnas de partición. El plan no constituye una medición del tiempo ahorrado.
